# 00 — Environment, GPU and the six-scenario smoke run

Provision a Colab/Kaggle GPU runtime, install the pinned toolchain, load
`Qwen/Qwen3-8B`, start the frozen gateway and run the **Stage A six-scenario smoke
set** through both the allow-all control and the defence — including per-scenario
saving and resume.

Every evaluation step is a call to a tested module or CLI
(`scripts/bench_campaign.py`, `scripts/bench_validate.py`, `scripts/bench_seal.py`,
`scripts/bench_score.py`, `benchmark/qwen.py`). No scoring, pairing or campaign
logic lives in a cell.

**No real-model result exists in this repository.** This notebook produces
engineering evidence only (Stage A licenses no research claim), and it commits no
output: run it in the cloud, keep the artifacts there.

Run this notebook from a checkout of the repository (see `notebooks/README.md`); it locates the repository root automatically and `chdir`s into it. Nothing is
committed: the run directories and bundles stay in the cloud.


In [ ]:
import hashlib, json, os, re, subprocess, sys, tempfile, time, urllib.request, zipfile
from pathlib import Path


def _repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "scripts" / "bench_campaign.py").is_file() or (
            candidate / "scripts" / "bench_validate.py"
        ).is_file():
            return candidate
    return start


REPO = Path(os.environ.get("AEGISGRAPH_REPO") or _repo_root(Path.cwd())).resolve()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

SCRATCH = Path(os.environ.get("AEGISGRAPH_SCRATCH") or tempfile.mkdtemp(prefix="aegisgraph-notebook-"))
SCRATCH.mkdir(parents=True, exist_ok=True)
ISSUER_DIR = SCRATCH / "dev-issuer"
POLICY_ADMIN_TOKEN = SCRATCH / "policy-admin.jwt"
DECISION_TOKEN = SCRATCH / "decision.jwt"
GATEWAY = {"proc": None, "url": None, "ablation": None}


def sh(args, *, check=True, capture=False, env=None):
    """Run a command from the repository root and echo it.

    ``capture=True`` returns the completed process with its output printed, so a
    notebook cell never silently swallows a CLI's diagnostics.
    """
    printable = [str(item) for item in args]
    print("$", " ".join(printable))
    proc = subprocess.run(printable, cwd=str(REPO), text=True, capture_output=capture, env=env)
    if capture:
        if proc.stdout:
            print(proc.stdout)
        if proc.stderr:
            print(proc.stderr, file=sys.stderr)
    if check and proc.returncode != 0:
        raise RuntimeError("command failed (exit %d): %s" % (proc.returncode, " ".join(printable)))
    return proc


def run_dirs_from(output):
    """The run directories a campaign invocation printed."""
    return [Path(match) for match in re.findall(r"run directory:\s*(\S+)", output)]


def init_issuer():
    if not (ISSUER_DIR / "jwks.json").is_file():
        sh([sys.executable, "scripts/dev_issuer.py", "init", "--directory", ISSUER_DIR], capture=True)
    for subject, role, path in (
        ("bench-policy-admin", "policy_admin", POLICY_ADMIN_TOKEN),
        ("bench-decision-client", "decision_client", DECISION_TOKEN),
    ):
        minted = sh(
            [sys.executable, "scripts/dev_issuer.py", "mint", "--directory", ISSUER_DIR,
             "--tenant", "bench", "--subject", subject, "--role", role, "--ttl", "7200"],
            capture=True,
        ).stdout
        path.write_text(json.loads(minted)["token"], encoding="utf-8")
    print("issuer ready:", ISSUER_DIR)


def stop_gateway():
    proc = GATEWAY.get("proc")
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=15)
        except Exception:
            proc.kill()
    GATEWAY.update(proc=None, url=None, ablation=None)


def start_gateway(port=8091, ablation=None):
    """Start the frozen gateway on loopback; optionally as an ablated build.

    The campaign CLI never manages processes and never sets AEGISGRAPH_*, so the
    notebook owns the gateway lifecycle. An ablation is a process-level switch.
    """
    stop_gateway()
    env = dict(os.environ)
    env.update({
        "PYTHONPATH": "backend",
        "AEGISGRAPH_AUTH_MODE": "required",
        "AEGISGRAPH_JWT_ISSUER": "https://dev-issuer.aegisgraph.local",
        "AEGISGRAPH_JWT_AUDIENCE": "aegisgraph",
        "AEGISGRAPH_JWKS": str(ISSUER_DIR / "jwks.json"),
    })
    if ablation:
        env["AEGISGRAPH_ABLATION"] = ablation
    log_path = SCRATCH / ("gateway-%s.log" % (ablation or "defence"))
    log = log_path.open("w", encoding="utf-8")
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "aegisgraph.app:app",
         "--host", "127.0.0.1", "--port", str(port)],
        cwd=str(REPO), env=env, stdout=log, stderr=subprocess.STDOUT,
    )
    url = "http://127.0.0.1:%d" % port
    GATEWAY.update(proc=proc, url=url, ablation=ablation)
    for _ in range(120):
        if proc.poll() is not None:
            raise RuntimeError("gateway exited early; see %s" % log_path)
        try:
            with urllib.request.urlopen(url + "/healthz", timeout=2) as response:
                if json.load(response).get("status") == "ok":
                    print("gateway ready at", url, "(ablation: %s)" % (ablation or "none"))
                    return url
        except Exception:
            time.sleep(0.5)
    raise TimeoutError("gateway did not become healthy; see %s" % log_path)


def publish_policies(url):
    sh([sys.executable, "scripts/bench_policies.py", "publish",
        "--defense-url", url, "--token-file", POLICY_ADMIN_TOKEN], capture=True)


def adapter_json(**overrides):
    """The frozen QwenConfig as the driver's inline --adapter-json value.

    The CLI takes the JSON itself (``--adapter-json '{"revision": ...}'``), not a
    path; a copy is also written to the scratch directory for the record.
    """
    from benchmark.qwen import QwenConfig
    payload = json.dumps(QwenConfig(**overrides).to_json(), sort_keys=True)
    (SCRATCH / "adapter.json").write_text(payload + "\n", encoding="utf-8")
    print(payload)
    return payload


def campaign(stage, condition, seeds, *, defense_url=None, adapter_json=None,
             resume=False, bundle=False, max_scenarios=None, dry_run=False):
    """Invoke the campaign CLI. The CLI owns all evaluation logic."""
    args = [sys.executable, "scripts/bench_campaign.py", "--stage", stage,
            "--condition", condition, "--model", MODEL, "--runs-dir", str(RUNS_DIR)]
    for seed in seeds:
        args += ["--seed", str(seed)]
    if defense_url:
        args += ["--defense-url", defense_url]
    if dry_run:
        args += ["--dry-run"]
    else:
        if MODEL == "qwen" and adapter_json is not None:
            args += ["--adapter-json", str(adapter_json)]
        if DECISION_TOKEN.is_file():
            args += ["--auth-token-file", str(DECISION_TOKEN)]
        if resume:
            args += ["--resume"]
        if max_scenarios is not None:
            args += ["--max-scenarios", str(max_scenarios)]
        if bundle:
            args += ["--bundle"]
    return sh(args, capture=True)


def score_run(run_dir):
    """Score a run directory into score.json/score.txt (protocol section 8)."""
    run_dir = Path(run_dir)
    table = sh([sys.executable, "scripts/bench_score.py", "--run", run_dir], capture=True).stdout
    (run_dir / "score.txt").write_text(table, encoding="utf-8")
    sh([sys.executable, "scripts/bench_score.py", "--run", run_dir, "--json",
        "--out", str(run_dir / "score.json")], capture=True)
    print("scored:", run_dir.name)


def verify_zip(zip_path):
    zip_path = Path(zip_path)
    digest = hashlib.sha256(zip_path.read_bytes()).hexdigest()
    with zipfile.ZipFile(zip_path) as archive:
        corrupt = archive.testzip()
    print("bundle:", zip_path.name, "sha256:", digest, "integrity:", "ok" if corrupt is None else corrupt)
    return digest


In [ ]:
# @title Parameters
STAGE = "A"
SEED = 1729                       # Stage A uses exactly one seed
MODEL = "qwen"
RUNS_DIR = Path(os.environ.get("AEGISGRAPH_RUNS_DIR", str(SCRATCH / "runs")))
DATASET = REPO / "benchmark" / "data"
GATEWAY_PORT = 8091
DEVICE = "cuda"

# Frozen inference configuration (freeze block section 3). Close the
# TBD-BEFORE-STAGE-B values from the Stage A record; "unpinned" is the honest value
# until then and is never replaced by an invented hash.
MODEL_REVISION = os.environ.get("AEGISGRAPH_MODEL_REVISION", "unpinned")
QUANTIZATION = "4bit"
DTYPE = "bfloat16"
TEMPERATURE = 0.0
TOP_P = 1.0
MAX_NEW_TOKENS = 768
THINKING = False
print("runs directory:", RUNS_DIR)


## 1. Runtime and GPU check

Report the interpreter, platform and GPU. The environment is *suitable* only when
a CUDA device with enough memory for the 4-bit model is visible to torch.


In [ ]:
import platform

print("python:", sys.version.split()[0])
print("platform:", platform.platform())
print("machine:", platform.machine())
try:
    import psutil
    ram_gb = psutil.virtual_memory().total / 1024 ** 3
    print("RAM (GiB):", round(ram_gb, 1))
except Exception as error:
    print("RAM: unknown (psutil not importable:", error, ")")

GPU_NAME, VRAM_GB = None, None
try:
    import torch
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        GPU_NAME, VRAM_GB = props.name, props.total_memory / 1024 ** 3
        print("GPU:", GPU_NAME, "| VRAM (GiB):", round(VRAM_GB, 1),
              "| devices:", torch.cuda.device_count())
        print("torch:", torch.__version__, "| cuda:", torch.version.cuda)
    else:
        print("no CUDA device is visible to torch")
except Exception as error:
    print("torch is not importable here:", error)

ENV_OK = GPU_NAME is not None and VRAM_GB is not None and VRAM_GB >= 14.0
print("environment suitable for the real run:", ENV_OK)


## 2. Install the pinned toolchain

The exact fail-closed command is a constant in `benchmark/qwen.py`, so the
notebook never hard-codes versions.


In [ ]:
if ENV_OK:
    import shlex
    from benchmark.qwen import INSTALL_COMMAND
    command = shlex.split(INSTALL_COMMAND)
    command[0] = sys.executable
    sh(command, check=False)
else:
    print("skipping the toolchain install: the environment is not suitable (section 1).")


## 3. Load `Qwen/Qwen3-8B`

`benchmark.qwen.QwenModelAdapter.load()` resolves the toolchain and fails closed
naming the exact install or download command; the manifest records the resolved
torch and transformers versions.


In [ ]:
ADAPTER_OK = False
if ENV_OK:
    from benchmark.qwen import QwenConfig, QwenModelAdapter
    adapter = QwenModelAdapter(QwenConfig(
        revision=MODEL_REVISION, quantization=QUANTIZATION, dtype=DTYPE,
        temperature=TEMPERATURE, top_p=TOP_P, max_new_tokens=MAX_NEW_TOKENS,
        thinking=THINKING, device=DEVICE,
    ))
    adapter.load()
    print(json.dumps(adapter.identity().to_json(), indent=2, sort_keys=True))
    ADAPTER_OK = True
else:
    print("skipping the model load: the environment is not suitable (section 1).")


## 4. Validate the dataset and verify the seal

Both are preconditions: the validator must end in `RESULT: PASS`, and the seal
must match its manifest **without** being opened.


In [ ]:
validated = sh([sys.executable, "scripts/bench_validate.py"], capture=True).stdout
if "RESULT: PASS" not in validated:
    raise RuntimeError("the dataset did not validate; the run must not start")
sh([sys.executable, "scripts/bench_seal.py", "status"], capture=True)
seal = sh([sys.executable, "scripts/bench_seal.py", "verify"], capture=True).stdout
if '"ciphertext_matches": true' not in seal:
    raise RuntimeError("the seal did not verify; the run must not start")


## 5. Gateway and policy sets on loopback

The dev issuer is a development-only identity (never a production IdP); the token
files are written UTF-8 without a BOM and are never printed. The gateway runs
from this checkout on loopback only.


In [ ]:
init_issuer()
GATEWAY_URL = start_gateway(GATEWAY_PORT)
publish_policies(GATEWAY_URL)


In [ ]:
# The frozen adapter configuration the driver records (--adapter-json).
ADAPTER_JSON = adapter_json(
    revision=MODEL_REVISION, quantization=QUANTIZATION, dtype=DTYPE,
    temperature=TEMPERATURE, top_p=TOP_P, max_new_tokens=MAX_NEW_TOKENS,
    thinking=THINKING, device=DEVICE,
)


## 6. Stage A smoke — the six-scenario set

First the dry run (no socket, no directory, no model), then the allow-all control
and the defence. Stage A uses exactly one seed.


In [ ]:
plan = campaign(STAGE, "defence", [SEED], defense_url=GATEWAY_URL, dry_run=True)
if "RESULT: DRY-RUN" not in plan.stdout:
    raise RuntimeError("the campaign CLI did not accept the Stage A plan")


In [ ]:
CONTROL_RUNS = []
if ADAPTER_OK:
    control = campaign(STAGE, "control", [SEED], defense_url=GATEWAY_URL)
    if "RESULT: REFUSED" in (control.stdout + control.stderr):
        raise RuntimeError("the control arm was refused")
    CONTROL_RUNS = run_dirs_from(control.stdout)
    print("control run directories:", CONTROL_RUNS)
else:
    print("skipping the real control run: the environment is not suitable (section 1).")


In [ ]:
if ADAPTER_OK:
    defence = campaign(STAGE, "defence", [SEED], defense_url=GATEWAY_URL, adapter_json=ADAPTER_JSON)
    DEFENCE_RUNS = run_dirs_from(defence.stdout)
    print("defence run directories:", DEFENCE_RUNS)
else:
    DEFENCE_RUNS = []
    print("skipping the real defence run: the environment is not suitable (section 1).")


## 7. Per-scenario saves and resume

A clean stop after two scenarios leaves an incomplete run directory; the resume
then continues it from the verified per-scenario checkpoints. A completed run
directory is never overwritten.


In [ ]:
if ADAPTER_OK:
    partial = campaign(STAGE, "defence", [SEED], defense_url=GATEWAY_URL,
                       adapter_json=ADAPTER_JSON, max_scenarios=2)
    if "RESULT: CAMPAIGN INCOMPLETE" not in partial.stdout:
        raise RuntimeError("expected a clean early stop from --max-scenarios")
    RESUMED_RUNS = run_dirs_from(partial.stdout)
    for run_dir in RESUMED_RUNS:
        checkpoints = sorted((run_dir / "checkpoints").glob("*.json"))
        print(run_dir.name, "checkpoints so far:", len(checkpoints))
    resumed = campaign(STAGE, "defence", [SEED], defense_url=GATEWAY_URL,
                       adapter_json=ADAPTER_JSON, resume=True)
    if "RESULT: CAMPAIGN COMPLETE" not in resumed.stdout and "RESULT: CAMPAIGN INCOMPLETE" not in resumed.stdout:
        raise RuntimeError("the resume did not report a terminal status")
    print(resumed.stdout)
else:
    print("skipping the resume exercise: the environment is not suitable (section 1).")


## 8. Score the runs

The driver never scores; scoring is `scripts/bench_score.py` (protocol section 8),
producing `score.json` and `score.txt` beside the raw outcomes.


In [ ]:
for run_dir in CONTROL_RUNS + DEFENCE_RUNS:
    score_run(run_dir)


## 9. Bundle the runs

`--bundle` is a standalone mode: it re-finds the newest completed run dir for
each (model, stage, condition, seed), extends `hashes.sha256` to cover the score
files and writes `<runs-dir>/<run-name>.zip`. The tarball is immutable — a second
bundle of the same run is refused.


In [ ]:
for condition in ["control", "defence"]:
    bundled = campaign(STAGE, condition, [SEED], defense_url=GATEWAY_URL, bundle=True)
    if "RESULT: BUNDLE OK" not in bundled.stdout:
        raise RuntimeError("bundling failed for %s" % condition)
    for zip_path in re.findall(r"zip:\s*(\S+)", bundled.stdout):
        verify_zip(Path(zip_path))


## 10. Unsuitable environment — the failure path

This cell is where a run that cannot succeed stops, with the exact reason rather
than a plausible-looking action. It passes when the environment is suitable.


In [ ]:
if not ENV_OK:
    raise RuntimeError(
        "this environment cannot run the real-model campaign: "
        + ("no CUDA device is visible to torch" if GPU_NAME is None else
           "the visible GPU has %.1f GiB, below the 14 GiB this 4-bit run needs" % VRAM_GB)
        + ". Provision a Colab/Kaggle GPU runtime (a free T4 is 15 GiB) and re-run; "
          "a scripted run (`--model scripted`) only checks the wiring, it is not a model run."
    )
if not ADAPTER_OK:
    raise RuntimeError("the Qwen adapter did not load; see the fail-closed message above.")
print("environment and adapter are ready; the smoke run has produced its artifacts.")


## 11. Teardown

Stop the gateway (the campaign CLI never manages processes). The run directories
under `RUNS_DIR` are the artifacts; download them before the session ends.


In [ ]:
stop_gateway()
print("run directories are under:", RUNS_DIR)
print("artifacts are not committed: keep them in the cloud and download them explicitly.")
